In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import os


In [ ]:
dataset_handle = "sergionefedov/language-learning-to-fluency-who-gets-there"

# Download the latest dataset
dataset_path = kagglehub.dataset_download(dataset_handle)
print("Dataset downloaded to:", dataset_path)

# List files included in the dataset
import os

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))


In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

dic = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sergionefedov/language-learning-to-fluency-who-gets-there",
    "data_dictionary_language.csv",
)
display(dic)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "sergionefedov/language-learning-to-fluency-who-gets-there",
    "language_learning_fluency.csv",
)

print("First 5 records:")
print(df.head())


In [ ]:
print(df.info)

In [ ]:
display(df.head())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
 

#Setting value to be predicted (target) and traits on which to base it (features)
target = df['cefr_level']
display(target)

features = df.drop(columns= ['learner_id','reached_fluency','cefr_level'])



display(features)

In [ ]:
#Normalizing values
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2, random_state=42,
    stratify=target)
numeric_cols = features_train.select_dtypes(include="number").columns
scaler = MinMaxScaler()


features_train[numeric_cols] = scaler.fit_transform(features_train[numeric_cols])
features_test[numeric_cols] = scaler.transform(features_test[numeric_cols])

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(features_train, target_train)

In [ ]:
log_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        random_state=42
    )
)

log_model.fit(features_train, target_train)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=7) #Seven language proficiency levels
)

knn_model.fit(features_train, target_train)


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hist_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    random_state=42
)

hist_model.fit(features_train, target_train)


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

extra_model = ExtraTreesClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

extra_model.fit(features_train, target_train)


In [ ]:
#Calculating how far off the false predictions are

cefr_order = {
    "A0": 0,
    "A1": 1,
    "A2": 2,
    "B1": 3,
    "B2": 4,
    "C1": 5,
    "C2": 6
}

true_numeric = target_test.map(cefr_order)
pred_numeric = pd.Series(target_pred, index=target_test.index).map(cefr_order)

mean_level_error = (true_numeric - pred_numeric).abs().mean()

print(f"Mean CEFR-level error: {mean_level_error:.2f}")

In [ ]:
#Testing multiple models
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


print("------randomforestclassifier-------------")
target_pred = model.predict(features_test)

print("Accuracy:", round(accuracy_score(target_test, target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, target_pred))

pred_numeric = pd.Series(target_pred, index=target_test.index).map(cefr_order)
mean_level_error = (true_numeric - pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {mean_level_error:.2f}")


print("\nConfusion matrix:")
print(confusion_matrix(target_test, target_pred))

print("------logistic regression-------------")
log_target_pred = log_model.predict(features_test) 


print("Accuracy:", round(accuracy_score(target_test, log_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, log_target_pred))

log_pred_numeric = pd.Series(log_target_pred, index=target_test.index).map(cefr_order)
log_mean_level_error = (true_numeric - log_pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {log_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, log_target_pred))

print("------KNN-------------")
knn_target_pred = knn_model.predict(features_test) 

print("Accuracy:", round(accuracy_score(target_test, knn_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, knn_target_pred))

knn_pred_numeric = pd.Series(knn_target_pred, index=target_test.index).map(cefr_order)
knn_mean_level_error = (true_numeric - knn_pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {knn_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, knn_target_pred))

print("------gradient boosting-------------")
hist_target_pred = hist_model.predict(features_test) 

print("Accuracy:", round(accuracy_score(target_test, hist_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, hist_target_pred))

hist_pred_numeric = pd.Series(hist_target_pred, index=target_test.index).map(cefr_order)
hist_mean_level_error = (true_numeric - pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {hist_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, hist_target_pred))

print("------extra trees-------------")
extra_target_pred = extra_model.predict(features_test) 

print("Accuracy:", round(accuracy_score(target_test, extra_target_pred),3))
print("\nClassification report:")
print(classification_report(target_test, extra_target_pred))

extra_pred_numeric = pd.Series(extra_target_pred, index=target_test.index).map(cefr_order)
extra_mean_level_error = (true_numeric - extra_pred_numeric).abs().mean()
print(f"Mean CEFR-level error: {extra_mean_level_error:.2f}")

print("\nConfusion matrix:")
print(confusion_matrix(target_test, extra_target_pred))

#Investigating which features were most influential in each model

In [ ]:
importance = permutation_importance(model, features_test, target_test, n_repeats=10, random_state=42)

importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(importance_df)


In [ ]:
log_importance = permutation_importance(log_model, features_test, target_test, n_repeats=10, random_state=42)

log_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": log_importance.importances_mean,
    "importance_std": log_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(log_importance_df)

In [ ]:
knn_importance = permutation_importance(knn_model, features_test, target_test, n_repeats=10, random_state=42)

knn_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": knn_importance.importances_mean,
    "importance_std": knn_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(knn_importance_df)

In [ ]:
hist_importance = permutation_importance(hist_model, features_test, target_test, n_repeats=10, random_state=42)

hist_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": hist_importance.importances_mean,
    "importance_std": hist_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(hist_importance_df)

In [ ]:
extra_importance = permutation_importance(extra_model, features_test, target_test, n_repeats=10, random_state=42)

extra_importance_df = pd.DataFrame({
    "feature": features_test.columns,
    "importance_mean": extra_importance.importances_mean,
    "importance_std": extra_importance.importances_std
}).sort_values("importance_mean", ascending=False)

display(extra_importance_df)